In [15]:
import pandas as pd

# 1. Veriyi yükle (Dosya adının aynı klasörde olduğundan emin ol)
df = pd.read_csv('YAP471_Cleaned_Close_Prices.csv')

# 2. Tarih sütununu indeks yap ki işlemler kolaylaşsın
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)

# 3. Günlük Getirileri (Daily Returns) hesapla
# pct_change() fonksiyonu (Bugün - Dün) / Dün işlemini otomatik yapar
returns = df.pct_change()

# 4. İlk satır boş kalacaktır (çünkü ilk günün bir öncesi yok), onu temizleyelim
returns = returns.dropna()

# Sonucu kontrol et
print("İlk 5 günlük getiri verisi:")
print(returns.head())

# İstersen bu sonuçları yeni bir dosya olarak kaydedebilirsin
# returns.to_csv('YAP471_Gunluk_Getiriler.csv')

İlk 5 günlük getiri verisi:
                AAPL      MSFT      AMZN     GOOGL      TSLA      NVDA
Date                                                                  
2021-03-16  0.012848  0.012365  0.003303  0.014333 -0.043872  0.007590
2021-03-17 -0.006502 -0.002848  0.014186 -0.000799  0.036831  0.003763
2021-03-18 -0.033915 -0.026623 -0.034353 -0.029238 -0.069322 -0.046424
2021-03-19 -0.004434 -0.001615  0.015509  0.002780  0.002618  0.009745
2021-03-22  0.028286  0.024501  0.011681  0.001852  0.023102  0.026487


In [16]:
# 5. 20 günlük hareketli (rolling) kovaryans matrisini hesapla
# Bu işlem, her tarih için hisseler arasındaki risk ilişkisini çıkarır
rolling_cov = returns.rolling(window=20).cov().dropna()

# Sonucu görmek için (Örneğin son tarihteki 6x6'lık matris)
print("\nEn güncel tarihteki Kovaryans Matrisi (Risk Tablosu):")
print(rolling_cov.tail(6))


En güncel tarihteki Kovaryans Matrisi (Risk Tablosu):
                      AAPL      MSFT      AMZN     GOOGL      TSLA      NVDA
Date                                                                        
2026-03-13 AAPL   0.000248  0.000055  0.000080  0.000045  0.000078  0.000222
           MSFT   0.000055  0.000198  0.000075 -0.000002  0.000127  0.000091
           AMZN   0.000080  0.000075  0.000261  0.000115  0.000189  0.000148
           GOOGL  0.000045 -0.000002  0.000115  0.000214  0.000098  0.000097
           TSLA   0.000078  0.000127  0.000189  0.000098  0.000343  0.000224
           NVDA   0.000222  0.000091  0.000148  0.000097  0.000224  0.000512


In [17]:
# 1. Zeynep'ten gelen dosyayı oku
z_scores = pd.read_csv('YAP471_ZScore_Expected_Returns.csv')
z_scores['Date'] = pd.to_datetime(z_scores['Date'])
z_scores.set_index('Date', inplace=True)

# 2. Dosya zaten 471_Project.ipynb'de z_scores * -1 ile kaydedildiği için
# burada tekrar negatif almıyoruz — bu sayede mean reversion sinyali korunuyor.
expected_returns = z_scores

# 3. Tarihleri eşitle (Verilerin çakıştığı ortak günleri bul)
common_dates = returns.index.intersection(expected_returns.index)
expected_returns = expected_returns.loc[common_dates]

print("\nBeklenen Getiri Sinyalleri (İlk 5 Gün):")
print(expected_returns.head())


Beklenen Getiri Sinyalleri (İlk 5 Gün):
                AAPL      MSFT      AMZN     GOOGL      TSLA      NVDA
Date                                                                  
2021-04-12 -1.794911 -1.842845 -2.102636 -1.484519 -1.316402 -2.519826
2021-04-13 -2.136748 -1.832478 -1.932636 -1.411248 -2.763378 -2.460098
2021-04-14 -1.449593 -1.356362 -1.236010 -1.159143 -1.659065 -1.722521
2021-04-15 -1.702196 -1.542771 -1.424865 -1.411819 -1.675836 -2.151362
2021-04-16 -1.455112 -1.484967 -1.416907 -1.249891 -1.497823 -1.694955


In [ ]:
import numpy as np

# ─────────────────────────────────────────────────────────────────────────────
# DÜZELTİLMİŞ OPTİMİZASYON FONKSİYONU  (scipy gerektirmez, pure numpy)
#
# Eski versiyondaki 3 temel sorun giderildi:
#
#  1) Negatif Beklenen Getiri Sorunu (TSLA'ya %100 yığılma):
#     → Negatif beklenen getiriler 0'a clip'lenir.
#       Mean reversion stratejisinde yalnızca 20-günlük MA'nın ALTINDA kalan
#       (oversold) hisseler "ucuz" kabul edilir ve pozitif beklenti alır.
#       MA üzerindeki (overbought) hisseler 0 beklenti alır, optimizer bu
#       hisselere ağırlık vermez.
#
#  2) Tek Hisseye Yığılma Sorunu (%100 TSLA):
#     → max_weight=0.40 sınırı: 2+ pozitif beklentili hisse varsa,
#       tek bir hisse portföyün %40'ından fazlasını alamaz.
#
#  3) Kovaryans Matrisinin Sayısal Kararsızlığı:
#     → Ridge regularization (+ 1e-8 * I) ile matris her zaman PD yapılır,
#       bu lineer cebir çözümünün yakınsamasını garantiler.
#
# ÇÖZÜM YÖNTEMİ: Analitik Max-Sharpe Portföyü = Σ⁻¹μ
#   Kısıtsız çözüm analitik olarak bulunur, ardından box constraint
#   (0 ≤ w_i ≤ max_weight) için iteratif projeksiyon uygulanır.
# ─────────────────────────────────────────────────────────────────────────────

def optimize_portfolio(exp_returns, cov_matrix, max_weight=0.40):
    n = len(exp_returns)

    # 1. Negatif beklenen getirileri sıfıra clip'le (mean reversion: sadece oversold)
    clipped = np.clip(exp_returns, 0, None)

    # 2. Tüm getiriler sıfır → piyasa overbought → defansif eşit ağırlık
    if clipped.sum() <= 1e-10:
        return np.ones(n) / n

    # 3. Ridge regularization ile kovaryans matrisini stabilize et
    reg_cov = cov_matrix + np.eye(n) * 1e-8

    # 4. Analitik çözüm: maksimum Sharpe portföyü ∝ Σ⁻¹μ
    try:
        inv_cov = np.linalg.inv(reg_cov)
        raw_weights = inv_cov @ clipped
    except np.linalg.LinAlgError:
        return np.ones(n) / n

    # 5. Negatif ağırlıkları sıfırla (long-only constraint)
    weights = np.clip(raw_weights, 0, None)
    if weights.sum() <= 1e-10:
        return np.ones(n) / n

    # 6. max_weight kısıtını iteratif projeksiyon ile uygula
    #    (Pozitif beklentili hisse sayısı 1 ise max_weight uygulanmaz)
    if (clipped > 1e-10).sum() > 1:
        for _ in range(200):
            weights = np.clip(weights, 0, max_weight)
            total = weights.sum()
            if total <= 1e-10:
                return np.ones(n) / n
            weights /= total
            if weights.max() <= max_weight + 1e-9:
                break

    weights /= weights.sum()
    return weights


# ─────────────────────────────────────────────────────────────────────────────
# Her tarih için optimizasyonu çalıştır
# ─────────────────────────────────────────────────────────────────────────────
portfolio_weights = {}

for date in common_dates:
    try:
        daily_ret = expected_returns.loc[date].values
        daily_cov = rolling_cov.loc[date].values
        weights = optimize_portfolio(daily_ret, daily_cov)
        portfolio_weights[date] = weights
    except KeyError:
        continue

weights_df = pd.DataFrame.from_dict(portfolio_weights, orient='index', columns=returns.columns)

print("Optimize Edilmiş Portföy Ağırlıkları (İlk 5 Gün):")
print(weights_df.head().round(4))
print()
print("Ortalama Ağırlık Dağılımı:")
print(weights_df.mean().round(4))
print()
single_days = (weights_df.max(axis=1) > 0.99).sum()
print(f"Tek hisseye yüklenen gün (oversold=1): {single_days}")
print(f"Bu günlerde hangi hisse seçildi:")
print(weights_df[weights_df.max(axis=1) > 0.99].idxmax(axis=1).value_counts().to_string())


In [19]:
# Sonuçları CSV olarak kaydet
weights_df.to_csv('YAP471_Optimize_Portfolio_Weights.csv')
print("\nDosya 'YAP471_Optimize_Portfolio_Weights.csv' adıyla kaydedildi.")


Dosya 'YAP471_Optimize_Portfolio_Weights.csv' adıyla kaydedildi.


In [20]:
import numpy as np

# Sinyal bugün hesaplanır, pozisyon ertesi gün açılır.
# Yani bugünkü ağırlıklar → yarınki getiriyi yakala.
# Bu yaklaşım look-ahead bias'ı önler.
weights_shifted = weights_df.shift(1)

# Ağırlık olan günlerle returns'u hizala
aligned_returns = returns.reindex(weights_shifted.index)

# Her gün için portföy getirisi = ağırlıkların getirilere göre ağırlıklı toplamı
portfolio_returns = (weights_shifted * aligned_returns).sum(axis=1)

# İlk gün shift'ten dolayı NaN gelir, onu at
portfolio_returns = portfolio_returns.dropna()

# Birikimli (kümülatif) getiri: başlangıçta 1 TL olsaydı ne olurdu?
cumulative_returns = (1 + portfolio_returns).cumprod()

print("Günlük Portföy Getirileri (İlk 5 Gün):")
print(portfolio_returns.head().round(4))
print()
print(f"Toplam işlem günü : {len(portfolio_returns)}")
print(f"Başlangıç tarihi  : {portfolio_returns.index[0].date()}")
print(f"Bitiş tarihi      : {portfolio_returns.index[-1].date()}")
print()
print(f"Ortalama günlük getiri : {portfolio_returns.mean():.4%}")
print(f"Günlük getiri std dev  : {portfolio_returns.std():.4%}")
print()
print(f"Kümülatif getiri (1 TL → ?): {cumulative_returns.iloc[-1]:.4f} TL")


Günlük Portföy Getirileri (İlk 5 Gün):
2021-04-13    0.0000
2021-04-14   -0.0395
2021-04-15    0.0090
2021-04-16    0.0013
2021-04-19   -0.0340
dtype: float64

Toplam işlem günü : 1236
Başlangıç tarihi  : 2021-04-13
Bitiş tarihi      : 2026-03-13

Ortalama günlük getiri : 0.0447%
Günlük getiri std dev  : 2.4077%

Kümülatif getiri (1 TL → ?): 1.2138 TL


In [21]:
# Kural: Her hisse için pozisyona giriş fiyatını takip et.
# Eğer o hissenin fiyatı giriş noktasından %10 düşerse,
# o günden itibaren o hissenin ağırlığını 0'a çek
# ve ağırlığı diğer aktif hisseler arasında yeniden dağıt.
#
# "Pozisyona giriş" = weights_df'de o hisse için ağırlığın
# ilk kez 0'dan büyük olduğu gün.

# Fiyat verisini al (orijinal kapanış fiyatları)
prices = df.copy()

# weights_df'yi kopyala; stop-loss sonrası ağırlıkları burada güncelleyeceğiz
weights_sl = weights_df.copy()

for ticker in weights_sl.columns:
    entry_price = None       # Pozisyona giriş fiyatı
    in_position = False      # O anda pozisyonda mıyız?
    stopped_out = False      # Stop-loss tetiklendi mi?

    for date in weights_sl.index:
        weight = weights_sl.loc[date, ticker]
        price = prices.loc[date, ticker] if date in prices.index else None

        if price is None:
            continue

        # Pozisyona yeni giriş: önceki gün 0'dı, bugün > 0
        if not in_position and weight > 1e-6:
            entry_price = price
            in_position = True
            stopped_out = False

        # Pozisyondaysak stop-loss kontrolü yap
        if in_position and entry_price is not None:
            drawdown = (price - entry_price) / entry_price
            if drawdown <= -0.10:
                # Stop-loss tetiklendi: bu hissenin ağırlığını 0 yap
                weights_sl.loc[date, ticker] = 0.0
                stopped_out = True
                in_position = False
                entry_price = None

        # Ağırlık 0'a düştüyse pozisyon kapandı demektir
        if in_position and weight < 1e-6:
            in_position = False
            entry_price = None

# Stop-loss sonrası ağırlık satırları toplamı 1'den küçük kalabilir.
# Kalan aktif ağırlıkları normalize ederek toplamı tekrar 1'e getir.
row_sums = weights_sl.sum(axis=1)
# Tüm ağırlıklar 0 olan satırlar (tam cash): normalize etme, 0 bırak
weights_sl = weights_sl.div(row_sums.where(row_sums > 1e-6, other=1), axis=0)

# Stop-loss sonrası portföy getirilerini yeniden hesapla
weights_sl_shifted = weights_sl.shift(1)
aligned_returns_sl = returns.reindex(weights_sl_shifted.index)
portfolio_returns_sl = (weights_sl_shifted * aligned_returns_sl).sum(axis=1).dropna()
cumulative_returns_sl = (1 + portfolio_returns_sl).cumprod()

print("Stop-Loss Öncesi vs Sonrası Karşılaştırması:")
print(f"  Stop-loss öncesi kümülatif getiri : {cumulative_returns.iloc[-1]:.4f} TL")
print(f"  Stop-loss sonrası kümülatif getiri: {cumulative_returns_sl.iloc[-1]:.4f} TL")
print()
print(f"  Stop-loss öncesi ort. günlük getiri : {portfolio_returns.mean():.4%}")
print(f"  Stop-loss sonrası ort. günlük getiri: {portfolio_returns_sl.mean():.4%}")
print()

# Kaç kez stop-loss tetiklendi?
stops_triggered = (weights_sl < weights_df - 1e-6).sum().sum()
print(f"Toplam stop-loss tetiklenme sayısı: {int(stops_triggered)}")


Stop-Loss Öncesi vs Sonrası Karşılaştırması:
  Stop-loss öncesi kümülatif getiri : 1.2138 TL
  Stop-loss sonrası kümülatif getiri: 1.1271 TL

  Stop-loss öncesi ort. günlük getiri : 0.0447%
  Stop-loss sonrası ort. günlük getiri: 0.0388%

Toplam stop-loss tetiklenme sayısı: 25


In [22]:
# Kural: Portföyün kümülatif değeri, ulaştığı en yüksek
# noktadan (peak) %15 veya daha fazla düşerse, tüm
# ağırlıkları o günden itibaren 0 yap (tamamen cash).
# Portföy, bir sonraki gün tekrar %5 yukarı çıktığında
# (recovery sinyali) yeniden pozisyona gir.

# Stop-loss uygulanmış getirileri temel al (Görev 2'den devam)
cumulative_sl = (1 + portfolio_returns_sl).cumprod()

# Drawdown kontrolüyle nihai ağırlıkları oluştur
weights_final = weights_sl.copy()

peak = 1.0          # Başlangıç değeri
in_cash = False     # Cash modunda mıyız?

for i, date in enumerate(portfolio_returns_sl.index):
    current_value = cumulative_sl.loc[date]

    # Peak'i güncelle
    if current_value > peak:
        peak = current_value

    # Drawdown hesapla
    drawdown = (current_value - peak) / peak

    if not in_cash:
        # %15 eşiği aşıldı mı?
        if drawdown <= -0.15:
            in_cash = True
            print(f"  ⚠ Drawdown limiti aşıldı! Tarih: {date.date()} | Drawdown: {drawdown:.2%} → Cash'e geçildi")
    else:
        # Recovery: peak'ten %5 toparlanma sinyali
        recovery = (current_value - (peak * 0.85)) / (peak * 0.85)
        if recovery >= 0.05:
            in_cash = False
            peak = current_value   # Peak'i sıfırla
            print(f"  ✓ Toparlanma sinyali! Tarih: {date.date()} → Pozisyona geri dönüldü")

    # Cash modundayken tüm ağırlıkları sıfırla
    if in_cash and date in weights_final.index:
        weights_final.loc[date] = 0.0

# Nihai portföy getirilerini hesapla (stop-loss + drawdown kuralı birlikte)
weights_final_shifted = weights_final.shift(1)
aligned_returns_final = returns.reindex(weights_final_shifted.index)
portfolio_returns_final = (weights_final_shifted * aligned_returns_final).sum(axis=1).dropna()
cumulative_returns_final = (1 + portfolio_returns_final).cumprod()

print()
print("=" * 55)
print("Risk Kontrolü Özeti:")
print(f"  Ham strateji kümülatif getiri       : {cumulative_returns.iloc[-1]:.4f} TL")
print(f"  + Stop-loss sonrası kümülatif getiri: {cumulative_returns_sl.iloc[-1]:.4f} TL")
print(f"  + Drawdown kuralı sonrası getiri    : {cumulative_returns_final.iloc[-1]:.4f} TL")
print("=" * 55)


  ⚠ Drawdown limiti aşıldı! Tarih: 2022-01-20 | Drawdown: -15.21% → Cash'e geçildi
  ✓ Toparlanma sinyali! Tarih: 2022-02-01 → Pozisyona geri dönüldü
  ⚠ Drawdown limiti aşıldı! Tarih: 2022-04-07 | Drawdown: -15.52% → Cash'e geçildi
  ✓ Toparlanma sinyali! Tarih: 2023-11-27 → Pozisyona geri dönüldü
  ⚠ Drawdown limiti aşıldı! Tarih: 2024-01-25 | Drawdown: -19.36% → Cash'e geçildi
  ✓ Toparlanma sinyali! Tarih: 2024-06-17 → Pozisyona geri dönüldü
  ⚠ Drawdown limiti aşıldı! Tarih: 2024-08-05 | Drawdown: -16.29% → Cash'e geçildi
  ✓ Toparlanma sinyali! Tarih: 2024-08-21 → Pozisyona geri dönüldü
  ⚠ Drawdown limiti aşıldı! Tarih: 2024-09-03 | Drawdown: -19.08% → Cash'e geçildi
  ✓ Toparlanma sinyali! Tarih: 2024-12-06 → Pozisyona geri dönüldü
  ⚠ Drawdown limiti aşıldı! Tarih: 2025-02-21 | Drawdown: -15.79% → Cash'e geçildi
  ✓ Toparlanma sinyali! Tarih: 2025-05-14 → Pozisyona geri dönüldü
  ⚠ Drawdown limiti aşıldı! Tarih: 2026-02-13 | Drawdown: -15.03% → Cash'e geçildi

Risk Kontrolü Öz

In [23]:
import numpy as np

TRADING_DAYS = 252  # Yıllık işlem günü sayısı
RISK_FREE_RATE = 0.05 / TRADING_DAYS  # Günlük risksiz faiz (%5 yıllık, ABD T-Bill baz)

def sharpe_ratio(returns, rf=RISK_FREE_RATE):
    # Yıllıklaştırılmış Sharpe Oranı
    excess = returns - rf
    if excess.std() == 0:
        return 0.0
    return (excess.mean() / excess.std()) * np.sqrt(TRADING_DAYS)

def max_drawdown(returns):
    # Maksimum Drawdown: Tepe noktasından en derin düşüş
    cumulative = (1 + returns).cumprod()
    rolling_peak = cumulative.cummax()
    drawdown = (cumulative - rolling_peak) / rolling_peak
    return drawdown.min()

def annualized_return(returns):
    # Yıllıklaştırılmış Getiri
    total = (1 + returns).prod()
    n_years = len(returns) / TRADING_DAYS
    return total ** (1 / n_years) - 1

def annualized_volatility(returns):
    # Yıllıklaştırılmış Volatilite
    return returns.std() * np.sqrt(TRADING_DAYS)

# Nihai strateji getirilerini değerlendir (stop-loss + drawdown kuralı uygulanmış)
sr    = sharpe_ratio(portfolio_returns_final)
mdd   = max_drawdown(portfolio_returns_final)
ann_r = annualized_return(portfolio_returns_final)
ann_v = annualized_volatility(portfolio_returns_final)

print("=" * 55)
print("  STRATEJİ PERFORMANS METRİKLERİ (Risk Kontrollü)")
print("=" * 55)
print(f"  Yıllıklaştırılmış Getiri    : {ann_r:.2%}")
print(f"  Yıllıklaştırılmış Volatilite: {ann_v:.2%}")
print(f"  Sharpe Oranı                : {sr:.4f}")
print(f"  Maksimum Drawdown           : {mdd:.2%}")
print(f"  Kümülatif Getiri            : {cumulative_returns_final.iloc[-1]:.4f} TL")
print("=" * 55)


  STRATEJİ PERFORMANS METRİKLERİ (Risk Kontrollü)
  Yıllıklaştırılmış Getiri    : -7.28%
  Yıllıklaştırılmış Volatilite: 24.36%
  Sharpe Oranı                : -0.3927
  Maksimum Drawdown           : -61.32%
  Kümülatif Getiri            : 0.6904 TL


In [24]:
# Stratejimiz hem buy-and-hold hem de
# 1/N (eşit ağırlık) baseline'ından daha yüksek Sharpe
# ve daha düşük max drawdown üretmeli.

# Strateji getirileriyle örtüşen tarih aralığını baz al
start = portfolio_returns_final.index[0]
end   = portfolio_returns_final.index[-1]
r     = returns.loc[start:end]

# --- BASELINE 1: Buy-and-Hold (6 hisseye eşit ağırlık, hiç değiştirme) ---
bh_weights = pd.Series([1/6] * 6, index=returns.columns)
bh_returns = r.dot(bh_weights)

# --- BASELINE 2: 1/N Rolling (her gün eşit ağırlık, günlük rebalance) ---
equal_weights = pd.DataFrame(
    [[1/6] * 6] * len(r),
    index=r.index,
    columns=returns.columns
)
equal_weights_shifted = equal_weights.shift(1).dropna()
aligned_r = r.reindex(equal_weights_shifted.index)
en_returns = (equal_weights_shifted * aligned_r).sum(axis=1)

# --- Metrikleri hesapla ---
def metrics(rets, label):
    sr  = sharpe_ratio(rets)
    mdd = max_drawdown(rets)
    ar  = annualized_return(rets)
    av  = annualized_volatility(rets)
    cum = (1 + rets).cumprod().iloc[-1]
    return {'Strateji': label, 'Yıllık Getiri': ar, 'Volatilite': av,
            'Sharpe Oranı': sr, 'Max Drawdown': mdd, 'Kümülatif': cum}

results = pd.DataFrame([
    metrics(portfolio_returns_final, 'Mean Reversion (Risk Kontrollü)'),
    metrics(bh_returns,              'Buy-and-Hold (Eşit Ağırlık)'),
    metrics(en_returns,              '1/N Rolling Rebalance'),
])
results.set_index('Strateji', inplace=True)

# Formatlı çıktı
print("=" * 70)
print("  BASELINE KARŞILAŞTIRMASI")
print("=" * 70)
fmt = results.copy()
fmt['Yıllık Getiri'] = fmt['Yıllık Getiri'].map('{:.2%}'.format)
fmt['Volatilite']    = fmt['Volatilite'].map('{:.2%}'.format)
fmt['Sharpe Oranı']  = fmt['Sharpe Oranı'].map('{:.4f}'.format)
fmt['Max Drawdown']  = fmt['Max Drawdown'].map('{:.2%}'.format)
fmt['Kümülatif']     = fmt['Kümülatif'].map('{:.4f} TL'.format)
print(fmt.to_string())
print("=" * 70)

# Strateji hedeflerini kontrol et
our_sr  = results.loc['Mean Reversion (Risk Kontrollü)', 'Sharpe Oranı']
our_mdd = results.loc['Mean Reversion (Risk Kontrollü)', 'Max Drawdown']
bh_sr   = results.loc['Buy-and-Hold (Eşit Ağırlık)', 'Sharpe Oranı']
bh_mdd  = results.loc['Buy-and-Hold (Eşit Ağırlık)', 'Max Drawdown']

print()
print("  HEDEF KONTROLÜ:")
print(f"  Sharpe > Buy-and-Hold? {'✓ EVET' if our_sr > bh_sr else '✗ HAYIR'} "
      f"({our_sr:.4f} vs {bh_sr:.4f})")
print(f"  Max DD < Buy-and-Hold? {'✓ EVET' if our_mdd > bh_mdd else '✗ HAYIR'} "
      f"({our_mdd:.2%} vs {bh_mdd:.2%})")
print("=" * 70)


  BASELINE KARŞILAŞTIRMASI
                                Yıllık Getiri Volatilite Sharpe Oranı Max Drawdown  Kümülatif
Strateji                                                                                     
Mean Reversion (Risk Kontrollü)        -7.28%     24.36%      -0.3927      -61.32%  0.6904 TL
Buy-and-Hold (Eşit Ağırlık)            24.38%     29.87%       0.7122      -47.13%  2.9157 TL
1/N Rolling Rebalance                  23.73%     29.85%       0.6948      -47.13%  2.8393 TL

  HEDEF KONTROLÜ:
  Sharpe > Buy-and-Hold? ✗ HAYIR (-0.3927 vs 0.7122)
  Max DD < Buy-and-Hold? ✗ HAYIR (-61.32% vs -47.13%)
